# **Dataset Selection**

In [ ]:
!pip install datasets==2.19.2 transformers==4.52.4 accelerate evaluate seqeval -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [ ]:
import datasets
import transformers

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)

datasets: 2.19.2
transformers: 4.52.4


In [ ]:
from datasets import load_dataset

dataset = load_dataset("eriktks/conll2003")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/datasets/load.py:1491: FutureWarning: The repository for eriktks/conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/eriktks/conll2003
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warni

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [ ]:
features = dataset["train"].features
print(features["pos_tags"])

Sequence(feature=ClassLabel(names=['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'], id=None), length=-1, id=None)


# **Data Preprocessing**

In [ ]:
!pip install transformers datasets seqeval evaluate accelerate -q

In [ ]:
!pip install conllu -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("universal_dependencies", "en_ewt")

train_dataset = dataset["train"]
test_dataset = dataset["test"]
val_dataset = dataset["validation"]

/usr/local/lib/python3.12/dist-packages/datasets/load.py:1491: FutureWarning: The repository for universal_dependencies contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/universal_dependencies
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Generating train split:   0%|          | 0/12543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2002 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2077 [00:00<?, ? examples/s]

# **Load Tokenizer**

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# **Create Label Mapping**

In [ ]:
label_names = train_dataset.features["upos"].feature.names

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

print(label_names)

['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']


# **Label Alignment Function**

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

# **Apply Preprocessing**

In [ ]:
tokenized_train = train_dataset.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_and_align_labels,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_and_align_labels,
    batched=True
)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_train[0]["input_ids"])
print(tokenized_train[0]["attention_mask"])
print(tokenized_train[0]["labels"])

[101, 2632, 1011, 23564, 2386, 1024, 2137, 2749, 2730, 21146, 28209, 14093, 2632, 1011, 2019, 2072, 1010, 1996, 14512, 2012, 1996, 8806, 1999, 1996, 2237, 1997, 1053, 4886, 2213, 1010, 2379, 1996, 9042, 3675, 1012, 102]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[-100, 10, 1, 10, -100, 1, 6, 0, 16, 10, -100, 10, 10, 1, 10, -100, 1, 8, 0, 2, 8, 0, 2, 8, 0, 2, 10, -100, -100, 1, 2, 8, 6, 0, 1, -100]


# **Model Setup**

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

print(model.config)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,
  "id2label": {
    "0": "NOUN",
    "1": "PUNCT",
    "2": "ADP",
    "3": "NUM",
    "4": "SYM",
    "5": "SCONJ",
    "6": "ADJ",
    "7": "PART",
    "8": "DET",
    "9": "CCONJ",
    "10": "PROPN",
    "11": "PRON",
    "12": "X",
    "13": "_",
    "14": "ADV",
    "15": "INTJ",
    "16": "VERB",
    "17": "AUX"
  },
  "initializer_range": 0.02,
  "label2id": {
    "ADJ": 6,
    "ADP": 2,
    "ADV": 14,
    "AUX": 17,
    "CCONJ": 9,
    "DET": 8,
    "INTJ": 15,
    "NOUN": 0,
    "NUM": 3,
    "PART": 7,
    "PRON": 11,
    "PROPN": 10,
    "PUNCT": 1,
    "SCONJ": 5,
    "SYM": 4,
    "VERB": 16,
    "X": 12,
    "_": 13
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinus

# **Training**

**Data Collator**

In [ ]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [ ]:
import evaluate

metric = evaluate.load("seqeval")

In [ ]:
import numpy as np

def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_names[p] for p, l in zip(prediction, label)
         if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_names[l] for p, l in zip(prediction, label)
         if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="pos_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=100
)

# **Trainer**

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
small_train = tokenized_train.shuffle(seed=42).select(range(3000))
small_val = tokenized_val.shuffle(seed=42).select(range(500))

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_1198/2171886930.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
print("dataset" in globals())
print("train_dataset" in globals())
print("tokenized_train" in globals())
print("model" in globals())
print("training_args" in globals())

True
True
True
True
True


In [ ]:
print("small_train" in globals())
print("small_val" in globals())
print("data_collator" in globals())
print("compute_metrics" in globals())

True
True
True
True


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

/tmp/ipykernel_1198/2171886930.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# **Train**

In [ ]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 3


wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.417300,0.277889,0.914528,0.910821,0.912671,0.926531
2,0.228800,0.213997,0.931265,0.931265,0.931265,0.942041
3,0.158800,0.202625,0.937037,0.936376,0.936707,0.946449


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADV seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: _ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: U

TrainOutput(global_step=564, training_loss=0.4284189356134293, metrics={'train_runtime': 89.4675, 'train_samples_per_second': 100.595, 'train_steps_per_second': 6.304, 'total_flos': 117754541071200.0, 'train_loss': 0.4284189356134293, 'epoch': 3.0})

# **Evaluation**

In [ ]:
results = trainer.evaluate(tokenized_test)

print(results)

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PRON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:17

{'eval_loss': 0.1949545443058014, 'eval_precision': 0.9342985352637372, 'eval_recall': 0.9382705552248959, 'eval_f1': 0.9362803325979977, 'eval_accuracy': 0.9465570804986433, 'eval_runtime': 10.2688, 'eval_samples_per_second': 202.263, 'eval_steps_per_second': 12.66, 'epoch': 3.0}


# **Inference**

In [ ]:
trainer.save_model("final_pos_model")
tokenizer.save_pretrained("final_pos_model")

('final_pos_model/tokenizer_config.json',
 'final_pos_model/special_tokens_map.json',
 'final_pos_model/vocab.txt',
 'final_pos_model/added_tokens.json',
 'final_pos_model/tokenizer.json')

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "token-classification",
    model="final_pos_model",
    tokenizer="final_pos_model",
    aggregation_strategy="simple"
)

Device set to use cuda:0


In [ ]:
sentence = "John works at Google in California"

results = classifier(sentence)

for item in results:
    print(item)

{'entity_group': 'PROPN', 'score': np.float32(0.9242653), 'word': 'john', 'start': 0, 'end': 4}
{'entity_group': 'VERB', 'score': np.float32(0.97244704), 'word': 'works', 'start': 5, 'end': 10}
{'entity_group': 'ADP', 'score': np.float32(0.9880846), 'word': 'at', 'start': 11, 'end': 13}
{'entity_group': 'PROPN', 'score': np.float32(0.9448304), 'word': 'google', 'start': 14, 'end': 20}
{'entity_group': 'ADP', 'score': np.float32(0.98696065), 'word': 'in', 'start': 21, 'end': 23}
{'entity_group': 'PROPN', 'score': np.float32(0.9692146), 'word': 'california', 'start': 24, 'end': 34}


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

sentence = "John works at Google in California"

results = classifier(sentence)

for r in results:
    print(r)

Device set to use cuda:0


{'entity_group': 'PROPN', 'score': np.float32(0.9242653), 'word': 'john', 'start': 0, 'end': 4}
{'entity_group': 'VERB', 'score': np.float32(0.97244704), 'word': 'works', 'start': 5, 'end': 10}
{'entity_group': 'ADP', 'score': np.float32(0.9880846), 'word': 'at', 'start': 11, 'end': 13}
{'entity_group': 'PROPN', 'score': np.float32(0.9448304), 'word': 'google', 'start': 14, 'end': 20}
{'entity_group': 'ADP', 'score': np.float32(0.98696065), 'word': 'in', 'start': 21, 'end': 23}
{'entity_group': 'PROPN', 'score': np.float32(0.9692146), 'word': 'california', 'start': 24, 'end': 34}


In [ ]:
results = trainer.evaluate()

print(results)

{'eval_loss': 0.20262549817562103, 'eval_precision': 0.937037037037037, 'eval_recall': 0.936376454000705, 'eval_f1': 0.936706629055007, 'eval_accuracy': 0.9464489795918367, 'eval_runtime': 2.1227, 'eval_samples_per_second': 235.552, 'eval_steps_per_second': 15.075, 'epoch': 3.0}


# **Report**

**Fine-Tuning DistilBERT for POS Tagging**

**Introduction**

Part-of-Speech (POS) Tagging is a token classification task where each word in a sentence is assigned a grammatical category such as noun,verb, adjective or pronoun Transformer models like DistilBERT can achieve high accuracy on POS tagging tasks through fine-tuning.


**Dataset**

The Universal Dependencies English EWT dataset was used It contains sentences annotated with Universal POS tags.

**Preprocessing**

The dataset was tokenized using the DistilBERT tokenizer Since transformers split words into subwords, label alignment was performed. Special tokens and extra subword tokens were assigned -100 to avoid affecting training.

**Model**

DistilBERT was fine-tuned using AutoModelForTokenClassification Label mappings were created using id2label and label2id.


**Evaluation**


The model was evaluated using the SeqEval metric Precision, Recall, F1 Score, and Accuracy were reported.


**Challenges Faced**


. Handling subword tokenization.

. Aligning labels correctly.

. Ignoring special tokens during training.

. Managing memory requirements during fine-tuning.

**Observations**

. DistilBERT achieved strong performance with limited training time.

. Proper label alignment significantly improved accuracy.

. Transformer models outperform traditional POS taggers on contextual understanding.

**POS Tagging vs Chunking**

POS tagging identifies the grammatical role of individual words, whereas chunking identifies phrase structures such as noun phrases and verb phrases Chunking is generally more complex because it requires phrase-level understanding.

**Conclusion**

Fine-tuning DistilBERT for POS tagging demonstrates the effectiveness of transformer-based token classification models for sequence labeling tasks.


